# Initialize

In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np

#For plotting
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

#For statistics
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr

import re
from Bio import SeqIO
import ast # for safe eveal, for parsing some of the data
import math

import const #to reload use import(importlib) and then importlib.reload(const)
from const import MPRA_data_paths
from const import pos_active_ctrl_color,neg_active_ctrl_color,highlight_color,custom_cmap
from const import set_equal_plot_limits
from const import plot_color_pallete
from const import custom_cmap_bolder
from const import FONT_SIZE_small
const.set_plot_style()
import matplotlib.ticker as mtick

#to reload consts without restarting kernel
#import importlib
#importlib.reload(const)

In [ ]:
# Use CPM normalization?
cpm=True

# Use log scale in visualization?
logScale=False

#Additional filters?
min_DNA_reads = 5 

### Import

In [ ]:
##Import oligos
library = "humanMPRA_L3a2" #humanMPRA_L3a2 humanMPRA_L1a1 humanMPRA_L1a1_Neurons #modern_humanMPRA_NPC modern_humanMPRA_Hob Max_MPRA_run2 PMID_38766054_Reilly thylacine_biorxiv_Gallego_Romero L4a3a4_NPCpresat
library_paths = MPRA_data_paths[library]

In [ ]:
#output directory
output_path = "/home/labs/davidgo/Collaboration/MPRA_QC_pipeline/activity/output/"+library+'/'
os.makedirs(output_path, exist_ok=True)

if "comb_df" in library_paths:
    print('loading activity_df...')
    activity_df = pd.read_csv(library_paths["comb_df"])
    
if "activity_per_rep" in library_paths:
    print('loading activity_per_rep...')
    activity_by_rep_df = pd.read_csv(library_paths["activity_per_rep"])

if "cDNA_reads_by_cell_type" in library_paths: # for PCA - activity_by_rep for two cell types
    print('loading cDNA_reads_by_cell_type...')
    if library == 'PMID_38766054_Reilly':
        cDNA_reads_by_cell_type_df = pd.read_csv(library_paths["cDNA_reads_by_cell_type"],sep = '\t')
    elif library == 'thylacine_biorxiv_Gallego_Romero':
        cDNA_reads_by_cell_type_df = pd.read_csv(library_paths["cDNA_reads_by_cell_type"],sep = '\t')        

if "UMI_counts" in library_paths:
    print('loading UMI_counts...')
    UMI_counts = pd.read_csv(library_paths["UMI_counts"],sep = '\t')
    #UMI_counts.rename(columns={'oligo_bc.1': 'oligo diversity'}, inplace=True)
    #UMI_counts['oligo diversity'] = 1/UMI_counts['oligo diversity']

if "oligo_fasta" in library_paths:
    print('loading oligo_fasta...')
    fasta_file = library_paths["oligo_fasta"]
    
if "different_std_threshold_analysis" in library_paths:
    print('loading different_std_threshold_analysis...')
    std_analysis_df = pd.read_csv(library_paths["different_std_threshold_analysis"],sep = '\t')
    
if "screen_df" in library_paths:
    print('loading screen_df...')
    screen_df = pd.read_csv(library_paths["screen_df"],index_col=0)
    
if "tss_df" in library_paths:
    print('loading tss_df...')
    distance_df = pd.read_csv(library_paths["tss_df"] ,index_col=0)
    
if "downsampling_activity_path" in library_paths:
    print("Activity downsampling data available")
    
if "downsampling_ratio_path" in library_paths:
    print("Ratio downsampling data available")

if "comparative_res" in library_paths:
    print('loading comparative_df...')
    comparative_df=pd.read_csv(library_paths["comparative_res"],index_col=0)

print('Done!')
                                                 
#hMPRA specific data
positive_ctrls_corr_dir = "/home/labs/davidgo/Collaboration/humanMPRA/additional/analyze_controls/chondrocytes/controls_df.csv"
positive_ctrls_corr_df = pd.read_csv(positive_ctrls_corr_dir)

### Create dictionary of postive controls, negative controls, highlighted oligos

In [ ]:
if library == 'Max_MPRA_run2':
    oligo_groups = ['POSITIVE_CONTROL','NEGATIVE_CONTROL','SLEA'] # NM 26-7-25 I have no idea what SLEA is, I just put it as a filler for the highlight oligos
else: 
    oligo_groups = ['NegCtrl_not_active','PosCtrl_osteoblast','PosCtrl_diff']

groups_dict = {}

for group in oligo_groups:
    compiled_pattern =  re.compile(group)
    group_oligos = set()
    
    with open(fasta_file, "r") as infile:
        for record in SeqIO.parse(infile, "fasta"):
            if compiled_pattern.search(record.id):
                group_oligos.add(record.id)
    groups_dict[group] = group_oligos

    
if library == 'Max_MPRA_run2':
    pos_olg = groups_dict['POSITIVE_CONTROL']  
    neg_olg = groups_dict['NEGATIVE_CONTROL']
    highlight_olg = groups_dict['SLEA']
elif library == 'PMID_38766054_Reilly':
    # Path to your Excel file
    supplementary_path = library_paths['supplementary']
    ctrl_oligo_seqs = pd.read_excel(supplementary_path, sheet_name="S23.MPRA Controls", skiprows=1)
    pos_olg = ctrl_oligo_seqs[ctrl_oligo_seqs["Control Type"] == "ctrl:pos"]["ID"].dropna().astype(str).tolist()
    neg_olg = ctrl_oligo_seqs[ctrl_oligo_seqs["Control Type"] == "ctrl:neg"]["ID"].dropna().astype(str).tolist()  
else: 
    pos_olg = groups_dict['PosCtrl_osteoblast']  
    neg_olg = groups_dict['NegCtrl_not_active']
    highlight_olg = groups_dict['PosCtrl_diff']

In [ ]:
# Add normalization
DNA_sum = activity_df["DNA_rep_comb"].sum()
RNA_sum = activity_df["RNA_rep_comb"].sum()

activity_df["DNA_rep_comb_cpm"] = 1000000*(activity_df["DNA_rep_comb"]+1)/DNA_sum
activity_df["RNA_rep_comb_cpm"] = 1000000*(activity_df["RNA_rep_comb"]+1)/RNA_sum

activity_df["DNA_rep_comb_cpm_log"] = np.log2(activity_df["DNA_rep_comb_cpm"])
activity_df["RNA_rep_comb_cpm_log"] = np.log2(activity_df["RNA_rep_comb_cpm"])


activity_df = activity_df[activity_df["DNA_rep_comb"] >= min_DNA_reads]

DNA_counts = "DNA_rep_comb"
RNA_counts = "RNA_rep_comb"

if cpm:
    DNA_counts=DNA_counts+"_cpm"
    RNA_counts=RNA_counts+"_cpm"

if logScale:
    DNA_counts=DNA_counts+"_log"
    RNA_counts=RNA_counts+"_log"

# Evaluating DNA and RNA complexity

### Retained cCREs and barcodes

In [ ]:
if "activity_per_rep" in library_paths:
    activity_by_rep_df_vectorized = activity_by_rep_df[['RNA_filtered_std2_rep1','DNA_filtered_std2_rep1',
                                                       'RNA_filtered_std2_rep2','DNA_filtered_std2_rep2',
                                                       'RNA_filtered_std2_rep3','DNA_filtered_std2_rep3']]
    def safe_eval(x):
        if pd.isna(x):   # catches np.nan, pd.NA, None
            return np.nan
        try:
            return ast.literal_eval(str(x))
        except (ValueError, SyntaxError):
            return x  # fallback: return as-is if it cannot be parsed

    activity_by_rep_df_vectorized = activity_by_rep_df_vectorized.applymap(safe_eval)
else:
    print("Essential data for the anlaysis is missing for this library")


In [ ]:
if "activity_per_rep" in library_paths:
    results = {}

    for col in activity_by_rep_df_vectorized.columns:
        print(col)
        series = activity_by_rep_df_vectorized[col]

        # drop NAs and ensure all entries are lists
        series = series.dropna().apply(lambda x: np.asarray(x, dtype=float))

        # 1. Fraction of rows with at least one non-zero
        frac_rows_nonzero = (series.apply(lambda arr: np.any(arr != 0))).mean()

        # 2. Flatten everything into one list and compute fraction non-zero
        if len(series) > 0:
            flat = np.concatenate([np.atleast_1d(x) for x in series.values])
            frac_values_nonzero = np.count_nonzero(flat) / flat.size
        else:
            frac_values_nonzero = np.nan

        results[col] = {
            "cCRE": frac_rows_nonzero,
            "Barcode": frac_values_nonzero,
        }

    results_df = pd.DataFrame(results).T

    # reset index so column names become a column
    results_df = results_df.reset_index().rename(columns={"index": "rep"})

    # Extract measurement type (before "_rep")
    results_df["measurement"] = results_df["rep"].str.split("_rep").str[0]


    # Melt into tidy format
    results_melted = results_df.melt(
        id_vars=["rep", "measurement"],
        value_vars=["cCRE", "Barcode"],
        var_name="metric",
        value_name="fraction"
    )

    print(results_melted)

else:
    print("Essential data for the anlaysis is missing for this library")


In [ ]:
if "activity_per_rep" in library_paths:

    print('NOTE: need to fix the x axis in illustrator')
    # Clean measurement name
    results_melted["measurement_clean"] = results_melted["measurement"].str.replace("_filtered_std2", "")

    # Build combined label with replicate number
    results_melted["group_rep"] = (
        results_melted["metric"] + " " +
        results_melted["measurement_clean"] + " " +
        results_melted["rep"].str.extract(r'(rep\d+)')[0]
    )

    # Desired order (12 bars total, clustered by metric)
    order = [
        "cCRE DNA rep1",
        "cCRE DNA rep2",
        "cCRE DNA rep3",
        "cCRE RNA rep1",
        "cCRE RNA rep2",
        "cCRE RNA rep3",
        "Barcode DNA rep1",
        "Barcode DNA rep2",
        "Barcode DNA rep3",
        "Barcode RNA rep1",
        "Barcode RNA rep2",
        "Barcode RNA rep3"
    ]

    # Assign colors: map each group to x or y
    palette = {grp: (plot_color_pallete['cCRE'] if "cCRE" in grp else plot_color_pallete['barcode']) for grp in order}

    plt.figure(figsize=(12,6))
    ax = sns.barplot(
        data=results_melted,
        x="group_rep",
        y="fraction",
        order=order,
        palette=palette,
        ci=None
    )


    # Add value labels above bars
    for patch in ax.patches:
        height = patch.get_height()
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            height + 0.02,  # a little above the bar
            f"{height*100:.0f}%",  # no digits after decimal
            ha="center",
            va="bottom",
            fontweight="bold",
            fontsize=FONT_SIZE_small
        )

    plt.ylabel("Percentage retained")
    plt.xlabel("")
    plt.yticks([0,1])

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    const.save_fig(plt,'retained_cCREs_and_barcodes',output_path)

    plt.show()

else:
    print("Essential data for the anlaysis is missing for this library")

### Reads per UMI

In [ ]:
if "UMI_counts" in library_paths:
    plt.clf()

    plt.hist(data=UMI_counts,x='oligo diversity', color='grey',bins = 55)
    plt.xlim(1,5)
    #plt.xticks(np.arange(1, 6, 1))
    plt.xticks([1,5])

    yticks = plt.gca().get_yticks()
    plt.yticks([yticks[0], yticks[-1]])

    #plt.yscale('log')
    plt.xlabel('Reads per UMI')
    plt.ylabel('Number of barcodes')


    const.save_fig(plt,'UMI_complexity_histogram',output_path)

    plt.show()

else:
    print("Essential data for the anlaysis is missing for this library")

### Activity distribution

In [ ]:
if "comb_df" in library_paths:

    plt.clf()

    bin_edges = np.linspace(-4, 4, 201)  # 100 bins between -10 and 20

    # First histogram (all scores)
    plt.hist(
        data=activity_df,
        x='ratio_log_rep_comb', bins=bin_edges, color=plot_color_pallete['default_color'], label='Non-active cCREs')

    # Second histogram (filtered scores)
    plt.hist(
        data=activity_df[activity_df['activity_adjusted'] == 'active'],
        x='ratio_log_rep_comb', bins=bin_edges, color='red', label='Active cCREs')

    plt.xlabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$')
    plt.ylabel('Number of cCREs')
    plt.legend()
    stat,pval=stats.skewtest(activity_df['ratio_log_rep_comb'].dropna())

    # Smallest positive float in Python
    min_p = np.nextafter(0, 1)  # ~5e-324 for float64

    if pval < min_p:
        p_text = f"p < {min_p:.1e}"
    else:
        p_text = f"p = {pval:.3g}"

    plt.text(
        0.05, 0.95,  # relative position in axes coordinates
        f"Skew test:\nstat = {stat:.2f}\n{p_text}",
        ha='left', va='top',
        transform=plt.gca().transAxes,
        fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
    )


    plt.xticks([-4,0,4])

    yticks = plt.gca().get_yticks()
    plt.yticks([yticks[0], yticks[-1]])

    plt.legend(
        loc="upper right",
        fontsize=10,         # smaller text
        markerscale=0.8,    # shrink the color boxes
        handlelength=1,     # shorter legend lines
        handletextpad=0.5,  # less padding between box and text
        borderpad=0.3       # tighter box
    )


    const.save_fig(plt,'activity_distribution',output_path)
    plt.show()

else:
    print("Essential data for the anlaysis is missing for this library")


### P-value distribution

In [ ]:
if "comb_df" in library_paths:
    # Drop NAs and sort p-values
    pvals = activity_df['pval.mad'].dropna().sort_values().to_numpy()
    n = len(pvals)

    # Expected quantiles under Uniform(0,1)
    expected = np.linspace(0, 1, n, endpoint=False) + 0.5/n   # mid-point adjustment

    plt.figure(figsize=(6,6))
    plt.scatter(expected, pvals, s=10, color='black', alpha=0.6,linewidths=0.1)
    plt.plot([0,1], [0,1], color='black', linestyle='--')  # y=x line

    plt.xticks([0,1])
    plt.yticks([0,1])


    plt.xlabel("Expected p-values\n (Uniform(0,1))")
    plt.ylabel("Observed p-values")
    plt.tight_layout()

    const.save_fig(plt,'P_value_distribution',output_path)

    plt.show()

else:
    print("Essential data for the anlaysis is missing for this library")

### P-value distribution - log

In [ ]:
if "comb_df" in library_paths:
    # Drop NAs and sort p-values
    pvals = activity_df['pval.mad'].dropna().sort_values().to_numpy()
    pvals_log = -np.log10(pvals)
    n = len(pvals)
    
    # Expected quantiles under Uniform(0,1)
    expected = np.linspace(0, 1, n, endpoint=False) + 0.5/n   # mid-point adjustment
    expected_log = -np.log10(expected)
    
    plt.figure(figsize=(6,6))
    plt.scatter(expected_log, pvals_log, s=10, color='black', alpha=0.6,linewidths=0.1)
    #plt.plot([0,1], [0,1], color='black', linestyle='--')  # y=x line
    plt.plot([0,np.max(np.array(expected_log)[np.isfinite(expected_log)])],
             [0,np.max(np.array(pvals_log)[np.isfinite(pvals_log)])],
             color='black', linestyle='--')  # y=x line

    #plt.xticks([0,1])
    #plt.yticks([0,1])


    plt.xlabel("Expected -log(p-values)\n (Uniform(0,1))")
    plt.ylabel("Observed -log(p-values)")
    plt.tight_layout()

    const.save_fig(plt,'P_value_distribution_log',output_path)

    plt.show()

else:
    print("Essential data for the anlaysis is missing for this library")

### Downsampling analysis - active cCREs

In [ ]:
if "downsampling_activity_path" in library_paths:
    act_perc_list=[]
    downsampling_perc_list=np.arange(0.1,1.01,0.1)
    for p in downsampling_perc_list:
        perc=round(p,1)
        print(perc)
        path=fr"{library_paths["downsampling_activity_path"]}/comb_df_adjusted_fdr_{perc}.csv"
        df=pd.read_csv(path)
        act_perc=(df['activity_adjusted']=='active').sum()/df.shape[0]
        act_perc_list.append(act_perc)
    
    summary_df=pd.DataFrame(data={"Sampling parameter":downsampling_perc_list, "% Active":act_perc_list})
    
    sc=sns.lineplot(data=summary_df,x='Sampling parameter',y='% Active',color=plot_color_pallete['cCRE'],marker="o")
    sc.set_xticks([0.1,1])
    sc.set_ylim(0,summary_df['% Active'].max())
    sc.set_yticks([sc.get_yticks()[0],sc.get_yticks()[-1]])
    sc.set_ylabel("Active cCREs (%)")
    sc.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
    const.save_fig(plt,'Activity downsampling',output_path)

## Downsampling analysis - correlation between replicates

In [ ]:
if "downsampling_activity_path" in library_paths and "downsampling_ratio_path" in library_paths:
    rep_corr_by_act=[]
    rep_corr_list=[]
    downsampling_perc_list=np.arange(0.1,1.01,0.1)
    summary_df=pd.DataFrame()
    for p in downsampling_perc_list:
        perc=round(p,1)
        print(perc)
        rep_path=fr"{library_paths["downsampling_ratio_path"]}/ratio_wo_outliers_std2_{perc}.csv"
        act_path = fr"{library_paths["downsampling_activity_path"]}/comb_df_adjusted_fdr_{perc}.csv"
        activity_by_rep_df_ds = pd.read_csv(rep_path)
        activity_df_ds=pd.read_csv(act_path)

        merged_df_ds=activity_by_rep_df_ds.merge(activity_df_ds,left_on='oligo',right_on='oligo',how='inner')
        merged_df_ds['activity_adjusted'].value_counts()

        x = merged_df_ds['ratio_log_filtered_std2_rep1']
        y = merged_df_ds['ratio_log_filtered_std2_rep2']
        act = merged_df_ds['activity_adjusted']
        df = pd.DataFrame({'x': x, 'y': y,'activity':act}).dropna()
        rep_corr_by_act.append(df.groupby('activity').apply(lambda g: pearsonr(g['x'],g['y'])[0],include_groups=False))
        rep_corr_list.append(pearsonr(df['x'],df['y'])[0])
    summary_df=pd.DataFrame(rep_corr_by_act)
    summary_df['all']=rep_corr_list
    summary_df['Sampling parameter']=downsampling_perc_list

    sc=sns.lineplot(data=summary_df,x='Sampling parameter',y='active',color='red',marker="o",label="Active")
    sns.lineplot(data=summary_df,x='Sampling parameter',y='non_active',color='gray',marker="o",label='Not active')
    sns.lineplot(data=summary_df,x='Sampling parameter',y='all',color='blue',marker="o",label='All')

    sc.set_xticks([0.1,1])
    sc.set_ylim(-1,1)
    sc.set_yticks([sc.get_yticks()[0],sc.get_yticks()[-1]])
    sc.set_ylabel("Correlation between replicates")
    plt.legend()

    const.save_fig(plt,'Correlation between replicates - downsampling',output_path)
    

### Cumulative RNA reads

In [ ]:
if "comb_df" in library_paths:
    activity_df['rna_counts_percentile']=activity_df['RNA_rep_comb'].rank(ascending=False)/activity_df.shape[0]
    sorted_activity_df=activity_df.sort_values(by='rna_counts_percentile')
    sorted_activity_df['rna_cumulative_sum']=sorted_activity_df['RNA_rep_comb'].cumsum()
    sorted_activity_df['rna_cumulative_percentile']=sorted_activity_df['rna_cumulative_sum']/sorted_activity_df['RNA_rep_comb'].sum()

else:
    print("Essential data for the anlaysis is missing for this library")

In [ ]:
if "comb_df" in library_paths:
    fig, ax = plt.subplots()
    ax.plot(sorted_activity_df['rna_counts_percentile'],
            sorted_activity_df['rna_cumulative_percentile'],
            color = plot_color_pallete['read'],
           linewidth=5)
    ax.set_xlabel('cCREs rank by RNA reads')
    ax.set_ylabel('Cumulative fraction of RNA reads')

    plt.xticks([0,1])
    plt.yticks([0,1])

    const.save_fig(plt,"cumulative_RNA_reads",output_path)

    plt.show()

else:
    print("Essential data for the anlaysis is missing for this library")

### DNA counts vs GC content (Bookdown only)

In [ ]:
if "oligo_fasta" in library_paths and "activity_df" in library_paths:
    #Processing
    # Initialize empty lists to store modified identifiers and sequences

    identifiers = []
    sequences = []
    gc_contents = []

    # Parse the FASTA file
    for record in SeqIO.parse(fasta_file, "fasta"):
        identifier = record.id  # Remove the first character
        identifiers.append(identifier)
        sequence = str(record.seq)
        sequences.append(sequence)

        # Calculate GC content
        gc_count = sequence.count('g') + sequence.count('c')+sequence.count('G') + sequence.count('C')
        total_bases = len(sequence)
        gc_content = (gc_count / total_bases) * 100
        gc_contents.append(gc_content)

    # Create a DataFrame
    gc_df = pd.DataFrame({'oligo': identifiers, 'Sequence': sequences, 'GC_Content': gc_contents})
    
else:
    print("Essential data for the anlaysis is missing for this library")


In [ ]:
if "oligo_fasta" in library_paths and "activity_df" in library_paths:
    # merge this to activity_df
    merged_gc_activity = activity_df.merge(gc_df, on = "oligo", how = "left")
    merged_gc_activity.loc[:,'DNA_rep_comb_log10'] = np.log10(merged_gc_activity['DNA_rep_comb'])
    merged_gc_activity.loc[:,'DNA_rep_comb_clipped_500'] = merged_gc_activity['DNA_rep_comb'].clip(upper=500, inplace=False)
    merged_gc_activity.loc[:, 'GC_Content_clipped_25_60'] = merged_gc_activity['GC_Content'].clip(lower=25, upper=60, inplace=False)

else:
    print("Essential data for the anlaysis is missing for this library")

In [ ]:
if "oligo_fasta" in library_paths and "activity_df" in library_paths:   
    # take oligos with specifc DNA count and compare high vs low gc
    bins = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
    labels = [
        '0-5', '5-10', '10-15', '15-20', '20-25', '25-30', '30-35', '35-40',
        '40-45', '45-50', '50-55', '55-60', '60-65', '65-70', '70-75', '75-80',
        '80-85', '85-90', '90-95', '95-100'
    ]


    merged_gc_activity.loc[:,'GC_Content_label'] = pd.cut(merged_gc_activity['GC_Content'], bins=bins, labels=labels, right=True)
else:
    print("Essential data for the anlaysis is missing for this library")

In [ ]:
if "oligo_fasta" in library_paths and "activity_df" in library_paths:   
    f, (ax_DNA ,ax_hist) = plt.subplots(2, sharex=True,gridspec_kw={"height_ratios": (.5,.5)})
    sns.boxplot(
        x=merged_gc_activity["GC_Content_label"],
        y=merged_gc_activity["DNA_rep_comb"],
        showfliers=False,
        color=plot_color_pallete['read'],
        ax=ax_DNA)

    # Set y-axis limits and ticks for DNA boxplot
    ax_DNA.set_ylim(bottom=0)
    yticks_DNA = ax_DNA.get_yticks()
    ax_DNA.set_yticks([yticks_DNA[0], yticks_DNA[-1]])

    sns.histplot(
        x=merged_gc_activity["GC_Content_label"],
        color=plot_color_pallete['cCRE'],
        ax=ax_hist)

    # Set x-axis limits to show 20-80% range
    # Find the positions of categories within the filtered data
    categories = [label for label in labels if label in merged_gc_activity["GC_Content_label"].cat.categories]
    first_cat_pos = 0
    last_cat_pos = len(categories) - 1

    ax_hist.set_xlim(-0.5, last_cat_pos + 0.5)
    ax_DNA.set_xlim(-0.5, last_cat_pos + 0.5)

    # Set x-axis ticks to show only "0" and "100"
    ax_hist.set_xticks([first_cat_pos, last_cat_pos])
    ax_hist.set_xticklabels(['0', '100'], rotation=45)

    ax_DNA.set_ylabel("DNA reads")
    ax_hist.set_xlabel("%GC")
    ax_hist.set_ylabel("cCREs")

    # Set y-axis limits and ticks for histogram
    ax_hist.set_ylim(bottom=0)
    yticks_hist = ax_hist.get_yticks()
    ax_hist.set_yticks([yticks_hist[0], yticks_hist[-1]])

    const.save_fig(plt,'GC_content_bias_in_DNA',output_path)
    plt.show()
else:
    print("Essential data for the anlaysis is missing for this library")

# Evaluating reproducability

### Similarity between samples (PCA) - Waiting for David

In [ ]:
def plot_rep_pca(counttable, groups):
    # Check that groups length matches number of samples
    if len(groups) != counttable.shape[1]:
        raise ValueError("Length of 'groups' must match number of columns in counttable")
    
    # Transpose for PCA (samples = columns in R)
    X = counttable.T.values
    X_scaled = StandardScaler().fit_transform(X)

    pca = PCA()
    pcs = pca.fit_transform(X_scaled)

    # Variance explained
    var_explained = pca.explained_variance_ratio_ * 100  # %

    # Build dataframe for plotting
    pca_df = pd.DataFrame(pcs, columns=[f"PC{i+1}" for i in range(pcs.shape[1])])
    pca_df["group"] = groups

    # Plot PC1 vs PC2
    plt.figure(figsize=(6, 5))
    ax = sns.scatterplot(
        data=pca_df,
        x="PC1",
        y="PC2",
        hue="group",
        style="group",
        s=80
    )

    # Axis labels with variance explained
    ax.set_xlabel(f"PC1 ({var_explained[0]:.1f}%)")
    ax.set_ylabel(f"PC2 ({var_explained[1]:.1f}%)")
    
    yticks = ax.get_yticks()
    ax.set_yticks([yticks[0], 0, yticks[-1]])
    
    xticks = ax.get_xticks()
    ax.set_xticks([xticks[0], 0, xticks[-1]])

    plt.tight_layout()
    plt.show()

In [ ]:
if "cDNA_reads_by_cell_type" in library_paths:
    if library == "PMID_38766054_Reilly":
        pca_input = cDNA_reads_by_cell_type_df[[
            "NA12878_r1", "NA12878_r2", "NA12878_r3","NA12878_r4","NA12878_r5",
            "HepG2_r1", "HepG2_r2", "HepG2_r3","HepG2_r4","HepG2_r5"
        ]]
        groups = [1,1,1,1,1,2,2,2,2,2]
    elif library == 'thylacine_biorxiv_Gallego_Romero':
        pca_input = cDNA_reads_by_cell_type_df[[
        "MC3T3_replicate1_cDNA", "MC3T3_replicate2_cDNA", "MC3T3_replicate3_cDNA",
        "O91_replicate1_cDNA", "O91_replicate2_cDNA", "O91_replicate3_cDNA"
        ]]
        groups = [1,1,1,2,2,2]

    # Check that groups length matches number of samples
    if len(groups) != pca_input.shape[1]:
        raise ValueError("Length of 'groups' must match number of columns in pca_input")

    # Transpose for PCA (samples = columns in R)
    X = pca_input.T.values
    X_scaled = StandardScaler().fit_transform(X)

    pca = PCA()
    pcs = pca.fit_transform(X_scaled)

    # Variance explained
    var_explained = pca.explained_variance_ratio_ * 100  # %

    # Build dataframe for plotting
    pca_df = pd.DataFrame(pcs, columns=[f"PC{i+1}" for i in range(pcs.shape[1])])
    pca_df["group"] = groups

    # Plot PC1 vs PC2
    plt.figure(figsize=(6, 5))
    ax = sns.scatterplot(
        data=pca_df,
        x="PC1",
        y="PC2",
        hue="group",
        style="group",
        s=80
    )

    # Rename legend labels
    new_labels = ["cell 1", "cell 2"]
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles[0:], labels=new_labels, title="Cell type")

    # Axis labels with variance explained
    ax.set_xlabel(f"PC1 ({var_explained[0]:.1f}%)")
    ax.set_ylabel(f"PC2 ({var_explained[1]:.1f}%)")

    # Remove tick labels for cleaner look
    ax.set_xticks([])
    ax.set_yticks([])

    plt.tight_layout()
    const.save_fig(plt, "PCA_similarity_between_samples", output_path)

else:
    print("Essential data for the anlaysis is missing for this library")

### Correlation between replicates

In [ ]:
if "activity_per_rep" in library_paths:
    # Prepare the data
    # Replace Inf values with NaN, then drop any rows with NaN values
    activity_by_rep_df = activity_by_rep_df.replace([np.inf, -np.inf], np.nan)

    # Drop rows where either 'ratio_log_filtered_std2_rep1' or 'ratio_log_filtered_std2_rep2' has NaN or Inf
    activity_by_rep_df = activity_by_rep_df.dropna(subset=['ratio_log_filtered_std2_rep1', 'ratio_log_filtered_std2_rep2'])


    x = activity_by_rep_df['ratio_log_filtered_std2_rep1'].values
    y = activity_by_rep_df['ratio_log_filtered_std2_rep2'].values


    # Create the KDE (Kernel Density Estimate)
    values = np.vstack([x, y])
    kernel = gaussian_kde(values)

    # Evaluate the KDE for each data point
    density = kernel(values)

else:
    print("Essential data for the anlaysis is missing for this library")

In [ ]:
if "activity_per_rep" in library_paths:
    max_density_threshold = 0.1 

    # Clip values in density before coloring
    density_capped = np.clip(density, a_min=None, a_max=max_density_threshold)

    plt.clf()

    scatter = plt.scatter(
        x, y,
        c=density_capped,      # use capped values
        cmap=custom_cmap_bolder,
        s=10, edgecolors='none'
    )

    plt.xlabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 1')
    plt.ylabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 2')

    #plt.xlim(-4, 6)
    #plt.ylim(-4, 6)
    xticks = plt.xticks()[0]
    yticks = plt.yticks()[0]
    plt.xticks([xticks[0], xticks[-1]])
    plt.yticks([yticks[0], yticks[-1]])

    const.save_fig(plt, 'RNA_DNA_ratio_correlation_between_replicates', output_path)
    plt.show()

else:
    print("Essential data for the anlaysis is missing for this library")

### Variation at various activity levels

In [ ]:
def merge_edge_bins(x, edges, min_count):
    """
    Merge only edge bins (leftmost and rightmost) into their neighbors 
    until each edge bin has at least `min_count` points.

    x      : 1D array-like of values
    edges  : 1D array-like of initial bin edges
    min_count : minimum number of points for edge bins
    """
    x = np.asarray(x)
    edges = np.asarray(edges, dtype=float)

    # Need at least 2 bins to do anything
    if edges.size <= 2:
        return edges

    while True:
        counts, _ = np.histogram(x, bins=edges)
        changed = False

        # --- left edge ---
        if len(counts) > 1 and counts[0] < min_count:
            # merge bin 0 with bin 1: remove the internal boundary edges[1]
            edges = np.delete(edges, 1)
            changed = True

        # recompute if we just changed edges
        counts, _ = np.histogram(x, bins=edges)

        # --- right edge ---
        if len(counts) > 1 and counts[-1] < min_count:
            # merge last bin with previous: remove internal boundary edges[-2]
            edges = np.delete(edges, -2)
            changed = True

        # stop when no more merges are needed / possible
        if not changed or edges.size <= 2:
            break

    return edges


def scale(n, n_max):
    return 30 + 200 * np.sqrt(n / n_max)

In [ ]:
if "activity_per_rep" in library_paths:
    
    merged_df=activity_by_rep_df.merge(activity_df,left_on='oligo',right_on='oligo',how='inner')
    merged_df['activity_adjusted'].value_counts()

    x = merged_df['ratio_log_filtered_std2_rep1']
    y = merged_df['ratio_log_filtered_std2_rep2']
    g = merged_df['activity_adjusted']
    df = pd.DataFrame({'x': x, 'y': y,'activity':g}).dropna()

    df["bin"] = None
    df['mask']=df['activity'].apply(lambda x: True if x=='active' else False)
    min_non_active=df.loc[~df['mask'],'x'].min()
    max_non_active=df.loc[~df['mask'],'x'].max()
    min_active=df.loc[df['mask'],'x'].min()
    max_active=df.loc[df['mask'],'x'].max()
    bins_non_active = np.linspace(np.floor(min_non_active), np.ceil(max_non_active), int((np.ceil(max_non_active)) - np.floor(min_non_active)))
    bins_active = np.linspace(np.floor(min_active), np.ceil(max_active), int((np.ceil(max_active)) - np.floor(min_active)))



    n_min = 30  # or whatever you decide

    # non-active
    x_non = df.loc[~df['mask'], 'x'].values
    bins_non_active_merged = merge_edge_bins(x_non, bins_non_active, n_min)
    df.loc[~df['mask'], 'bin'] = pd.cut(
        df.loc[~df['mask'], 'x'],
        bins=bins_non_active_merged,
        include_lowest=True
    )

    # active
    x_act = df.loc[df['mask'], 'x'].values
    bins_active_merged = merge_edge_bins(x_act, bins_active, n_min)
    df.loc[df['mask'], 'bin'] = pd.cut(
        df.loc[df['mask'], 'x'],
        bins=bins_active_merged,
        include_lowest=True
    )



    results = []
    for b, group in df.groupby('bin',observed=False):
        corr_func = spearmanr
        r, _ = corr_func(group['x'], group['y'])
        mean_y = np.mean(group['y'])
        std_y = np.std(group['y'], ddof=1)
        cv_y = (std_y / abs(mean_y))*100 if mean_y != 0 else np.nan
        mid = group['x'].median()
        active_flag=group['mask'].iloc[0]
        results.append({'bin': str(b), 'mid_x': mid, 'corr': r,'cv':cv_y,'n': len(group),'activity_flag':active_flag})

    corr_df = pd.DataFrame(results).sort_values('mid_x')
    n_max= corr_df['n'].max()
    sizes_act = scale(corr_df.loc[corr_df['activity_flag'],'n'], n_max)
    sizes_non_act = scale(corr_df.loc[~corr_df['activity_flag'],'n'], n_max)


    fig, ax1 = plt.subplots()

    ax1.plot(corr_df.loc[corr_df['activity_flag'],'mid_x'], corr_df.loc[corr_df['activity_flag'],'corr'], '-', lw=2, color='red')
    ax1.scatter(corr_df.loc[corr_df['activity_flag'],'mid_x'], corr_df.loc[corr_df['activity_flag'],'corr'],
                s=sizes_act, color='red', edgecolor='black', alpha=0.8, zorder=3)

    ax1.plot(corr_df.loc[~corr_df['activity_flag'],'mid_x'], corr_df.loc[~corr_df['activity_flag'],'corr'], '-', lw=2, color='gray')
    ax1.scatter(corr_df.loc[~corr_df['activity_flag'],'mid_x'], corr_df.loc[~corr_df['activity_flag'],'corr'],
                s=sizes_non_act, color='gray', edgecolor='black', alpha=0.8, zorder=3)

    ax1.set_xlabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 1')

    ax1.set_ylabel(f"Pearson's correlation (r)")
    ax1.axhline(0, color='gray', linestyle='--', lw=1)
    ax1.set_ylim(-1,1)



    example_ns = [corr_df['n'].min(), corr_df['n'].max()]
    example_ns = [int(v) for v in example_ns]


    ax1.set_yticks([ax1.get_yticks()[0], ax1.get_yticks()[-1]])
    x_ticks = ax1.get_xticks()
    xlim = ax1.get_xlim()
    visible_x_ticks = [t for t in x_ticks if xlim[0] <= t <= xlim[1]]

    ax1.set_xticks([visible_x_ticks[0],visible_x_ticks[-1]])



    size_handles = [
        ax1.scatter([], [], 
                    s=scale(circ, n_max),
                    facecolors='none',
                    edgecolors='black',
                    label=f"{circ}")
        for circ in [100, 1000, 10000]
    ]

    color_handles = [
        plt.Line2D([0], [0], color='red',  lw=2, label='Active'),
        plt.Line2D([0], [0], color='gray', lw=2, label='Non-active'),
    ]

    # Combine both types
    handles = size_handles + color_handles
    labels = [h.get_label() for h in handles]

    ax1.legend(handles=handles, labels=labels, title="Number of cCREs", frameon=False)

    fig.tight_layout()
    const.save_fig(plt,'Variation at various activity levels',output_path)
    plt.show()

### Correlation between replicates (controls)

In [ ]:
if "activity_per_rep" in library_paths:
    plt.clf()

    scatter = plt.scatter(
        x, y, c='lightgray', s=10, edgecolors='none')

    plt.xlabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 1')
    plt.ylabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 2')

    plt.scatter(data=activity_by_rep_df[activity_by_rep_df["oligo"].isin(neg_olg)],
                    x = 'ratio_log_filtered_std2_rep1',
                    y = 'ratio_log_filtered_std2_rep2',
                    s=15,label = 'negative control', color = neg_active_ctrl_color)

    plt.scatter(data=activity_by_rep_df[activity_by_rep_df["oligo"].isin(pos_olg)],
                    x = 'ratio_log_filtered_std2_rep1',
                    y = 'ratio_log_filtered_std2_rep2',
                    s=15,label = 'positive control', color = pos_active_ctrl_color)


    set_equal_plot_limits(x,y)

    #plt.xlim(-4, 6)
    #plt.ylim(-4, 6)
    plt.xticks([])
    plt.yticks([])  

    plt.legend()

    const.save_fig(plt,'RNA_DNA_ratio_correlation_between_replicates_with_controls',output_path)
    plt.show()

else:
    print("Essential data for the anlaysis is missing for this library")

## Activity of controls - sample comparison (always hMPRA chonds)

In [ ]:
#Requiers a specific dataset of control activity in various batches
positive_ctrls_corr_dir = "/home/labs/davidgo/Collaboration/humanMPRA/additional/analyze_controls/chondrocytes/controls_df.csv"
positive_ctrls_corr_df = pd.read_csv(positive_ctrls_corr_dir)

alpha = positive_ctrls_corr_df.filter(regex='alpha')

corr_alpha = alpha.corr()
plt.clf()
sns.heatmap(corr_alpha, cmap="Blues", annot=True)


const.save_fig(plt,'positive_controls_corr_between_libs_hMPRA',output_path)
plt.show()


## Outlier barcodes + min(DNA)

In [ ]:
if "different_std_threshold_analysis" in library_paths:

    # Define parameters
    outlier_filters = ["no_filter", "filtered_std3", "filtered_std2"]
    dna_thresholds = [0, 10, 25]
    reps = ['rep1', 'rep2', 'rep3']

    # Create subplot grid
    fig, axes = plt.subplots(len(dna_thresholds), len(outlier_filters))
    activity_level = 1
    max_density_threshold = 0.1

    for n, outlier_filter in enumerate(outlier_filters):
        for m, threshold in enumerate(dna_thresholds):

            # Mask by DNA threshold
            for rep in reps:
                std_analysis_df[f'ratio_{outlier_filter}_{rep}_DNA_{threshold}'] = (
                    std_analysis_df[f"ratio_log_{outlier_filter}_{rep}"]
                    .where(std_analysis_df[f"DNA_{outlier_filter}_sum_{rep}"] >= threshold, pd.NA)
                )

            # Drop NaNs before KDE
            df_plot = std_analysis_df.dropna(
                subset=[
                    f'ratio_{outlier_filter}_rep1_DNA_{threshold}',
                    f'ratio_{outlier_filter}_rep2_DNA_{threshold}'
                ]
            )
            x = df_plot[f'ratio_{outlier_filter}_rep1_DNA_{threshold}'].values
            y = df_plot[f'ratio_{outlier_filter}_rep2_DNA_{threshold}'].values

            # Compute density
            if len(x) > 1 and len(y) > 1:
                values = np.vstack([x, y])
                kernel = gaussian_kde(values)
                density = kernel(values)
                density_capped = np.clip(density, a_min=None, a_max=max_density_threshold)
            else:
                density_capped = np.zeros_like(x)

            # Plot correlation with density coloring
            axes[m, n].scatter(
                x, y,
                c=density_capped,
                cmap=custom_cmap_bolder,
                s=5,
                edgecolors='none'
            )

            # --- Square aspect ratio and limits ---
            axes[m, n].set_xlim(-3, 6.5)
            axes[m, n].set_ylim(-3, 6.5)
            axes[m, n].set_aspect('equal', adjustable='box')

            # Remove all ticks and tick labels
            axes[m, n].set_xticks([])
            axes[m, n].set_yticks([])

            # Label only outer edges
            if m == len(dna_thresholds) - 1:
                # Replace x-axis labels with descriptive text
                label_map = {
                    "no_filter": "No outlier removal",
                    "filtered_std3": "Outlier removal >3 SD",
                    "filtered_std2": "Outlier removal >2 SD"
                }
                axes[m, n].set_xlabel(label_map[outlier_filter])
            if n == 0:
                axes[m, n].set_ylabel(f'DNA≥{threshold}')

    plt.tight_layout()
    const.save_fig(plt,'minimizing_noise',output_path)
    plt.show()

else:
    print("Essential data for the analysis is missing for this library")


In [ ]:
if "different_std_threshold_analysis" in library_paths:

    # Define parameters
    outlier_filters = ["no_filter", "filtered_std3", "filtered_std2"]
    dna_thresholds = [0, 10, 25]
    reps = ['rep1', 'rep2', 'rep3']

    # Create subplot grid
    fig, axes = plt.subplots(len(dna_thresholds), len(outlier_filters), figsize=(12, 10))
    activity_level = 1  # <-- set or compute your activity threshold
    max_density_threshold = 0.1  # same as your example

    for n, outlier_filter in enumerate(outlier_filters):
        for m, threshold in enumerate(dna_thresholds):

            # Mask by DNA threshold
            for rep in reps:
                std_analysis_df[f'ratio_{outlier_filter}_{rep}_DNA_{threshold}'] = (
                    std_analysis_df[f"ratio_log_{outlier_filter}_{rep}"]
                    .where(std_analysis_df[f"DNA_{outlier_filter}_sum_{rep}"] >= threshold, pd.NA)
                )

            # Drop NaNs before KDE
            df_plot = std_analysis_df.dropna(
                subset=[
                    f'ratio_{outlier_filter}_rep1_DNA_{threshold}',
                    f'ratio_{outlier_filter}_rep2_DNA_{threshold}'
                ]
            )
            x = df_plot[f'ratio_{outlier_filter}_rep1_DNA_{threshold}'].values
            y = df_plot[f'ratio_{outlier_filter}_rep2_DNA_{threshold}'].values

            # Compute density
            if len(x) > 1 and len(y) > 1:
                values = np.vstack([x, y])
                kernel = gaussian_kde(values)
                density = kernel(values)
                density_capped = np.clip(density, a_min=None, a_max=max_density_threshold)
            else:
                density_capped = np.zeros_like(x)

            # Plot correlation with density coloring
            axes[m, n].scatter(
                x, y,
                c=density_capped,
                cmap=custom_cmap_bolder,
                s=5,
                edgecolors='none'
            )

            # --- Fixed axis limits and ticks ---
            axes[m, n].set_xlim(-3, 6.5)
            axes[m, n].set_ylim(-3, 6.5)
            axes[m, n].set_xticks([-3, 0, 3, 6])
            axes[m, n].set_yticks([-3, 0, 3, 6])

            # Hide ticks for inner subplots
            if n != 0:  # not leftmost column
                axes[m, n].set_yticklabels([])
                axes[m, n].set_ylabel("")
            if m != len(dna_thresholds) - 1:  # not bottom row
                axes[m, n].set_xticklabels([])
                axes[m, n].set_xlabel("")

            # Label axes only once per column/row
            if m == len(dna_thresholds) - 1:
                axes[m, n].set_xlabel(f'{outlier_filter}')
            if n == 0:
                axes[m, n].set_ylabel(f'DNA≥{threshold}')

    plt.tight_layout()
    plt.show()

else:
    print("Essential data for the anlaysis is missing for this library")

## Outlier barcodes

# Evaluating dynamic range and concordance with endogenous signals

## RNA vs DNA

In [ ]:
if "comb_df" in library_paths:

    # Prepare the data
    x = activity_df[DNA_counts].values
    y = activity_df[RNA_counts].values


    # Evaluate the KDE on the grid
    values = np.vstack([x, y])
    kernel = gaussian_kde(values)
    density = kernel(values)
    
else:
    print("Essential data for the anlaysis is missing for this library")

In [ ]:
if "comb_df" in library_paths:
    plt.clf()
    clip_threshold = 0.00002
    clipped_density = np.clip(density,0.00002, 0.0002)
    colors = ["#FEE090", "#FDAE61", "#F46D43", "#D73027", "#A50026"]
    custom_cmap = LinearSegmentedColormap.from_list("custom_cmap", colors, N=256)
    plt.scatter(x, y, cmap=custom_cmap, c=clipped_density, s=14, edgecolors='none')

    # Set axis limits
    plt.xlim(0, 250)
    plt.ylim(0, 250)

    # Show only extreme ticks
    plt.xticks([0, 250])
    plt.yticks([0, 250])

    #Set axis labels
    plt.xlabel('DNA count')
    plt.ylabel('RNA count')


    const.save_fig(plt,'RNA_DNA_ratio',output_path)
    plt.show()

else:
    print("Essential data for the anlaysis is missing for this library")

## Activity of Controls

In [ ]:
if "comb_df" in library_paths:
    
    #overrride control colors for this specific analysis
    pos_color = '#8FBCE1'
    neg_color = '#DE2326'
    high_color = '#BBA9D2'
    palette = [
        pos_color,  # 'PosCtrl_osteoblast_active'
        pos_color,  # 'PosCtrl_neuron_active'
        pos_color,  # 'PosCtrl_chondrocyte_active'
        pos_color,  # 'PosCtrl_diff'
        pos_color,  # 'NegCtrl_active_not_diff'
        neg_color,  # 'NegCtrl_not_active'
        neg_color,  # 'scrambled_control'
        neg_color,  # 'NegCtrl_non_SCREEN'
        high_color         # 'No control'
    ]


    if re.findall("a4",library):
        print(library)
        activity_df.loc[activity_df['oligo'].str.contains('posCtrlActive_BJ'), 'control_type'] = 'posCtrlActive_BJ'
        activity_df.loc[activity_df['oligo'].str.contains('posCtrlActive_adipocytes'), 'control_type'] = 'posCtrlActive_adipocytes'
        activity_df.loc[activity_df['oligo'].str.contains('posCtrlActive_NPC'), 'control_type'] = 'posCtrlActive_NPC'
        activity_df.loc[activity_df['oligo'].str.contains('posCtrlActive_HOb'), 'control_type'] = 'posCtrlActive_HOb'
        activity_df.loc[activity_df['oligo'].str.contains('NegCtrl_not_active'), 'control_type'] = 'NegCtrl_not_active'
        activity_df.loc[activity_df['oligo'].str.contains('scrambled_control'), 'control_type'] = 'scrambled_control'
        activity_df.loc[activity_df['oligo'].str.contains('PosCtrl_diff'), 'control_type'] = 'PosCtrl_diff'
        activity_df['control_type'] = activity_df['control_type'].fillna(value='No control')

        sns.boxplot(data=activity_df[~activity_df['control_type'].isin(('PosCtrl_diff','NegCtrl_active_not_diff'))],
                x="mad.score", y="control_type", showfliers = False,linewidth= 2,
                order=['posCtrlActive_BJ',
                       'posCtrlActive_HOb',
                       'posCtrlActive_adipocytes',
                       'posCtrlActive_NPC',
                       'NegCtrl_not_active',
                       'scrambled_control',
                       'No control'],
                palette=[pos_color,pos_color,pos_color,pos_color,neg_color,neg_color,high_color])
    elif library == 'archaic_MPRA':
        print(library)
        activity_df.loc[activity_df['oligo'].str.contains('posCtrlActive_BJ76'), 'control_type'] = 'PosCtrl_BJ'
        activity_df.loc[activity_df['oligo'].str.contains('NegCtrl_not_active'), 'control_type'] = 'NegCtrl_not_active'
        activity_df.loc[activity_df['oligo'].str.contains('scrambled_control'), 'control_type'] = 'scrambled_control'
        activity_df['control_type'] = activity_df['control_type'].fillna(value='No control')

        sns.boxplot(data=activity_df[~activity_df['control_type'].isin(('PosCtrl_diff','NegCtrl_active_not_diff'))],
                x="mad.score", y="control_type", showfliers = False,linewidth= 2,
                order=['PosCtrl_adipocyte_active',
                       'PosCtrl_neuron_active',
                       'PosCtrl_fibroblast_active',
                       'PosCtrl_osteoblast_active',
                       'NegCtrl_not_active',
                       'scrambled_control',
                      'No control'],
                palette=[pos_color,pos_color,pos_color,pos_color,neg_color,neg_color,high_color])

    elif library == 'Max_MPRA_run2':
        print(library)
        ax = sns.boxplot(
            data=activity_df,
            x="mad.score", y="control_type",
            showfliers=False, linewidth=2,
            order=['PosCtrl', 'NegCtrl', 'No control'],
            palette=[pos_color, neg_color, high_color]
        )

        ax.set_yticklabels(["Positive control", "Negative control", "cCREs"])


    elif library == 'PMID_38766054_Reilly':
        print(library)
        conditions = [
            activity_df["oligo"].isin(pos_olg),
            activity_df["oligo"].isin(neg_olg)
        ]
        choices = ["Positive control", "Negative control"]
        activity_df["control_type"] = np.select(conditions, choices, default="cCREs")

        sns.boxplot(data=activity_df,
                x="statistic", y="control_type", showfliers = False,linewidth= 2,
                order=['Positive control',
                       'Negative control',
                      'cCREs'],
                palette=[pos_color,neg_color,high_color])
    else:  
        activity_df.loc[activity_df['oligo'].str.contains('PosCtrl_osteoblast_active'), 'control_type'] = 'PosCtrl_osteoblast_active'
        activity_df.loc[activity_df['oligo'].str.contains('PosCtrl_chondrocyte_active'), 'control_type'] = 'PosCtrl_chondrocyte_active'
        activity_df.loc[activity_df['oligo'].str.contains('PosCtrl_neuron_active'), 'control_type'] = 'PosCtrl_neuron_active'
        activity_df.loc[activity_df['oligo'].str.contains('NegCtrl_non_SCREEN'), 'control_type'] = 'NegCtrl_non_SCREEN'
        activity_df.loc[activity_df['oligo'].str.contains('NegCtrl_active_not_diff'), 'control_type'] = 'NegCtrl_active_not_diff'
        activity_df.loc[activity_df['oligo'].str.contains('scrambled_control'), 'control_type'] = 'scrambled_control'
        activity_df.loc[activity_df['oligo'].str.contains('NegCtrl_not_active'), 'control_type'] = 'NegCtrl_not_active'
        activity_df.loc[activity_df['oligo'].str.contains('PosCtrl_diff'), 'control_type'] = 'PosCtrl_diff'
        activity_df['control_type'] = activity_df['control_type'].fillna(value='No control')

        label_map = {
            'PosCtrl_osteoblast_active': 'Positive control (osteoblast)',
            'PosCtrl_neuron_active': 'Positive control (neuron)',
            'PosCtrl_chondrocyte_active': 'Positive control (chondrocyte)',
            'NegCtrl_not_active': 'Negative control (inactive)',
            'scrambled_control': 'Negative control (scrambled)',
            'No control': 'cCREs'
        }

        # Apply the label map
        activity_df['control_type_pretty'] = activity_df['control_type'].map(label_map)

        # Plot with the new, readable names
        sns.boxplot(
            data=activity_df[~activity_df['control_type'].isin(('PosCtrl_diff','NegCtrl_active_not_diff'))],
            x="mad.score", y="control_type_pretty", showfliers=False, linewidth=2,
            order=[
                'Positive control (osteoblast)',
                'Positive control (neuron)',
                'Positive control (chondrocyte)',
                'Negative control (inactive)',
                'Negative control (scrambled)',
                'cCREs'
            ],
            palette=[
                pos_color,
                pos_color,
                pos_color,
                neg_color,
                neg_color,
                high_color
            ]
        )
    plt.xlabel("Activity statistic")
    plt.ylabel("")

    xticks = plt.gca().get_xticks()
    plt.xticks([xticks[0], xticks[-1]])

    const.save_fig(plt,'Control_activity_boxplots',output_path)

    plt.show()

else:
    print("Essential data for the anlaysis is missing for this library")

## Genomic annotations

In [ ]:
screen_ccre_colors = {
    "Promoter":     "#D63B30",  # strong red – promoter‐like signature
    "Proximal Enhancer":    "#D36728",  # dark orange – proximal enhancer‐like signature
    "Distal Enhancer":    "#F8BE35",  # gold/yellow – distal enhancer‐like signature
    "DNase-H3K4me3": "#8DBCE2",  # (light) blue – (novel promoters/poised enhancers)
    "DNase-only":   "#4383B6",  # grey – DNase only open chromatin
    "Heterochromatin":    "#D3D3D3"   # light grey – low DNase signal / inactive
}




if "screen_df" in library_paths:
    
    screen_df['mask']=screen_df['activity_adjusted'].apply(lambda x: True if x=='active' else False)
    qbins = pd.qcut(screen_df.loc[screen_df['mask'], 'statistic'],
                        q=5,
                        labels=[f"Q{i}" for i in range(1, 6)])
    screen_df['bin']='Inactive'
    screen_df.loc[screen_df['mask'],'bin']=qbins
    counts_df=pd.DataFrame(screen_df.groupby('bin')['class'].value_counts())
    counts_df=counts_df.reset_index()
    counts_df_wide=counts_df.pivot(index='bin',columns=['class'],values='count')
    counts_df_wide=counts_df_wide.reset_index()
    counts_df_wide_prop=counts_df_wide.iloc[:,1:].apply(lambda x: x/x.sum(),axis=1)
    counts_df_wide_prop['bin']=counts_df_wide['bin']
    bin_order = ['Inactive',"Q1","Q2","Q3","Q4","Q5"]
    counts_df_wide_prop["bin"] = pd.Categorical(counts_df_wide_prop["bin"], categories=bin_order, ordered=True)
    counts_df_wide_prop=counts_df_wide_prop.sort_values("bin")
    counts_df_wide_prop=counts_df_wide_prop[['Promoter','Proximal Enhancer','Distal Enhancer','DNase-H3K4me3','DNase-only','Heterochromatin','bin']]

    
    ax = counts_df_wide_prop.plot(
        x='bin', kind='bar', stacked=True,color=screen_ccre_colors
    )

    # move legend to the side
    ax.legend(title="Chromatin mark",
              loc="center left",          # anchor relative to axes bbox
              bbox_to_anchor=(1.02, 0.5), # x>1 pushes it outside to the right
              frameon=False)
    plt.xlabel('Activity quantile')
    formatter = mtick.PercentFormatter(xmax=1.0)
    ax.yaxis.set_major_formatter(formatter)
    ax.set_yticks([0,1])

    plt.ylabel('cCREs (%)')
    const.save_fig(plt,"Genomic annotation screen",output_path)

## Proximity to TSS

In [ ]:
if "tss_df" in library_paths:

    distance_df['mask']=distance_df['activity_adjusted'].apply(lambda x: True if x=='active' else False)
    qbins = pd.qcut(distance_df.loc[distance_df['mask'], 'statistic'],
                    q=5,
                    labels=[f"Q{i}" for i in range(1, 6)])
    distance_df['bin']='Inactive'
    distance_df.loc[distance_df['mask'],'bin']=qbins
    bin_order = ['Inactive',"Q1","Q2","Q3","Q4","Q5"]

    f, ax_box = plt.subplots(figsize=(4,8))
    sns.boxplot(data=distance_df,x="bin",y="log10_distance",showfliers=False,
                color=plot_color_pallete['cCRE'],ax=ax_box,order=bin_order,
               medianprops={"color": "#FFFFFF", "linewidth": 2})
    ax_box.set_ylabel(r"Distance from TSS (bp, $\mathbf{log_{2}}\!$)")
    ax_box.set_xlabel("Activity quantile")
    ax_box.tick_params(axis='x', labelrotation=90)

    ax_box.set_yticks([ax_box.get_yticks()[1],ax_box.get_yticks()[-1]])
    
    
    
    const.save_fig(plt,"Genomic annotation TSS",output_path)


 ## Correlation between AI predictions and MPRA results

## Differential activity distribution

In [ ]:
if "comparative_res" in library_paths:
    def round_down(num, dec=0):
        mult = 10 ** dec
        return math.floor(num * mult) / mult
    def round_up(num, dec=0):
        mult = 10 ** dec
        return math.ceil(num * mult) / mult
    
    max_lim=round_up(comparative_df['logFC'].max(),2)
    min_lim=round_down(comparative_df['logFC'].min(),2)
    bins = np.arange(min_lim,max_lim,0.05)

    comparative_df['logFC'].hist(color='gray',bins=bins, label="Active",grid=False)
    comparative_df.loc[comparative_df["differentialy_active"]==True, "logFC"].hist(
        color="red",
        bins=bins,
        label="Differentially active",
        grid=False)
    plt.xlabel("Fold Change, log2")
    plt.ylabel("#cCREs")
    #plt.title("Modern Human-derived MethMPRA results in osteoblasts")
    plt.legend(loc='best')
    const.save_fig(plt,"Differential activity distribution",output_path)
